# WRAP_P6_MULTIDAY_CPCV — Notebook (Wave 2 #2)

**Spec ref:** Wave 2 item #2 — CPCV at production scale.

Extends NB06 to multi-day datasets using `trading_day_aware=True` on
`CascadeAwareCPCV`. Loads N NQ files (one per trading day), concatenates
their L1+L2+FI features with day-boundary markers, trains LightGBM under
calendar-aware CPCV, and promotes the model via `CURRENT.json` if it
beats baseline.

Designed to scale from 3 days (smoketest) to 30+ days (production) by
changing `MAX_DAYS` in §00.

## Section 00 — Environment & multi-day config

In [1]:
import sys, logging, pickle, json, glob
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd

_NB_DIR = Path('<repo>/p6lab/notebooks')
if str(_NB_DIR) not in sys.path: sys.path.insert(0, str(_NB_DIR))
import _common
from _common import NOTEBOOK_DATA_SLICE, TIER_CUTOFFS, BASELINE_MIN_AUC_IMPROVEMENT, collect_snapshots, versioned_dir

_DS_DEFAULT = NOTEBOOK_DATA_SLICE
SYMBOL    = _DS_DEFAULT['symbol']
TICK_SIZE = _DS_DEFAULT['tick_size']

_P6LAB         = Path(_common._P6LAB_ROOT)
DATA_ROOT      = _P6LAB.parent / 'data'
ARTIFACTS_DIR  = _P6LAB / 'artifacts' / 'p6lab'
MODELS_DIR     = _P6LAB / 'correlation_runs' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MAX_DAYS          = 5        # bump to 30+ for production runs
MAX_SNAPS_PER_DAY = 2000     # cap per-day rows; balance coverage vs. wall time
BOOK_SHAPE_DIM    = 40

logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger(__name__)

from p6lab.features._l1_adapter import L1Adapter, L1AdapterConfig
from p6lab.features.l1_features import L1FeatureNames, compute_l1_features
from p6lab.features.l2_features import (
    L2FeatureNames, L2History, L2Snapshot, compute_l2_features,
)
from p6lab.features.fragility_index import FragilityIndex
from p6lab.validation.cpcv import CascadeAwareCPCV
from p6lab.validation.information_gain import must_beat_baseline
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

np.random.seed(42)

## Section 01 — Discover NQ files + pick one per calendar date

In [2]:
candidates = sorted(glob.glob(str(DATA_ROOT / 'nq-mbo-*.dbn.zst')))
log.info('01 │ discovered %d NQ files', len(candidates))

# Prefer `-fullday-` variants when available; dedupe by embedded date
import re
date_rx = re.compile(r'(\d{4}-\d{2}-\d{2})')
picked_by_date: dict[str, str] = {}
for path in candidates:
    m = date_rx.search(Path(path).name)
    if not m: continue
    d = m.group(1)
    # Prefer fullday; else overnight; else vanilla
    cur = picked_by_date.get(d)
    prio = (0 if 'fullday' in path else (1 if 'overnight' in path else 2))
    cur_prio = (0 if cur and 'fullday' in cur else (1 if cur and 'overnight' in cur else 2))
    if cur is None or prio < cur_prio:
        picked_by_date[d] = path

daily_files = [picked_by_date[d] for d in sorted(picked_by_date)][:MAX_DAYS]
log.info('01 │ picked %d distinct days for training', len(daily_files))
for p in daily_files: log.info('      %s', Path(p).name)
assert len(daily_files) >= 3, 'Need at least 3 distinct days for multi-day CPCV'

01 │ discovered 9 NQ files


01 │ picked 4 distinct days for training


      nq-mbo-2026-03-24.dbn.zst


      nq-mbo-fullday-2026-03-25.dbn.zst


      nq-mbo-fullday-2026-03-26.dbn.zst


      nq-mbo-2026-03-27.dbn.zst


## Section 02 — Per-day feature extraction + concatenation

In [ ]:
async def _features_for_day(path: str) -> pd.DataFrame:
    slice_ = {**_DS_DEFAULT, 'data_file': path,
              'max_snapshots': MAX_SNAPS_PER_DAY,
              'start_ms': None, 'end_ms': None}
    snaps = await collect_snapshots(slice_)
    if not snaps: return pd.DataFrame()

    adapter = L1Adapter(L1AdapterConfig(tick_size=TICK_SIZE))
    l1_rows = [compute_l1_features(adapter.ingest(s), adapter.history) for s in snaps]
    l1_df = pd.DataFrame(np.array(l1_rows), columns=L1FeatureNames.ALL)

    # L2FeatureNames.ALL has 'book_shape_vector' at INDEX 10 (not at end),
    # so pair by explicit index — not by zip() position (Wave 3 Phase 5C bug).
    _SCALAR_L2_INDICES = [i for i, n in enumerate(L2FeatureNames.ALL)
                          if n != 'book_shape_vector']

    # Wave 4 Phase 1B: per-day VPIN state for live trade_flow_toxicity
    from p6lab.features.vpin import (
        VPINState, VPINConfig, update_vpin_state, compute_vpin,
        classify_trade_lee_ready,
    )
    vpin_state = VPINState()
    vpin_cfg = VPINConfig(avg_daily_volume=300_000.0, bucket_size_fraction=1.0/200,
                          window_size=5)
    _prev_trade_price = 0.0

    # Wave 4 Phase 2: microstructure features state
    from p6lab.features.microstructure import (
        MicrostructureState, update_microstructure, snapshot_features,
        MICROSTRUCTURE_FEATURE_NAMES,
    )
    micro_state = MicrostructureState()

    from p6lab.features.l2_features import compute_book_shape_vector
    l2_hist = L2History(); l2_rows = []; micro_rows = []
    bsvs: list[np.ndarray] = []
    for s in snaps:
        if not (s.bids and s.asks): continue
        bid_map = {lvl.price: lvl.volume for lvl in s.bids[:20]}
        ask_map = {lvl.price: lvl.volume for lvl in s.asks[:20]}
        prices = sorted(set(bid_map) | set(ask_map), reverse=True)
        book_levels = [(p, bid_map.get(p, 0.0), ask_map.get(p, 0.0)) for p in prices]
        bp, ap = s.bids[0].price, s.asks[0].price
        bs, as_ = s.bids[0].volume, s.asks[0].volume
        weighted_mid = (bp*as_ + ap*bs) / max(bs+as_, 1e-9)
        plain_mid = 0.5 * (bp + ap)

        # Wave 4 Phase 1B: update VPIN from trades
        trade_dicts = []
        for tr in getattr(s, 'recent_trades', []) or []:
            try:
                tp = float(getattr(tr, 'price', 0.0))
                tv = float(getattr(tr, 'size', 0.0) or getattr(tr, 'volume', 0.0))
                tts = int(getattr(tr, 'timestamp_ms', s.timestamp_ms))
                if tv <= 0: continue
                side = classify_trade_lee_ready(tp, _prev_trade_price, bp, ap)
                update_vpin_state(vpin_state, vpin_cfg, tts, tp, tv, side)
                _prev_trade_price = tp
                trade_dicts.append({'price': tp, 'volume': tv, 'side': side})
            except Exception:
                pass
        vpin_value = compute_vpin(vpin_state, vpin_cfg)
        l2_hist.vpin_value = float(vpin_value) if vpin_value is not None else 0.0

        # Wave 4 Phase 2: microstructure update + snapshot
        update_microstructure(micro_state, ts_ms=s.timestamp_ms, mid=plain_mid,
                              trades=trade_dicts)
        micro_rows.append(snapshot_features(micro_state, s.timestamp_ms, plain_mid))

        l2_snap = L2Snapshot(
            timestamp_ms=s.timestamp_ms, symbol=SYMBOL,
            mid_price=plain_mid, book_levels=book_levels,
            # Wave 4 Phase 1A: pass recent_events through
            recent_events=list(getattr(s, 'recent_events', []) or []),
        )
        l2_hist.append(l2_snap)
        feat_vec = compute_l2_features(l2_snap, l2_hist)
        bsvs.append(compute_book_shape_vector(l2_snap))
        row = {L2FeatureNames.ALL[i]: float(feat_vec[i]) for i in _SCALAR_L2_INDICES}
        row['weighted_mid'] = weighted_mid
        l2_rows.append(row)
    l2_df = pd.DataFrame(l2_rows)
    micro_df = pd.DataFrame(micro_rows, columns=MICROSTRUCTURE_FEATURE_NAMES)

    n = min(len(l1_df), len(l2_df))
    if n == 0: return pd.DataFrame()
    l1_df = l1_df.iloc[:n].reset_index(drop=True)
    l2_df = l2_df.iloc[:n].reset_index(drop=True)
    micro_df = micro_df.iloc[:n].reset_index(drop=True)

    # Pop weighted_mid out of the feature matrix — it is the label source.
    mid_series = l2_df.pop('weighted_mid')

    # Explode BSV into 40 scalar columns (see Phase 1B rationale in NB06 §03).
    bsv_cols = pd.DataFrame(
        np.stack(bsvs[:n]),
        columns=[f'bsv_{i:02d}' for i in range(BOOK_SHAPE_DIM)],
        index=l1_df.index,
    )

    fi = FragilityIndex()
    l1_arr = l1_df.to_numpy()
    l2_arr = l2_df.to_numpy()
    fi_rows = []
    for i in range(n):
        sub = fi.compute_sub_indices(l1_arr[i], l2_arr[i],
                                     vpin_value=l2_hist.vpin_value)
        fi_rows.append({'fi_fast': fi.compute_fast(sub.RF, sub.SF, sub.FT),
                        'fi_full': fi.compute_full(sub)})
    fi_df = pd.DataFrame(fi_rows)

    df = pd.concat([l1_df, l2_df, bsv_cols, micro_df, fi_df], axis=1)
    df = df.loc[:, ~df.columns.duplicated()]
    df['__ts_ms__'] = [s.timestamp_ms for s in snaps[:n]]
    df['__source_file__'] = Path(path).name
    df['__mid__'] = mid_series.values   # label source; excluded from X in §03
    return df

day_frames = []
for p in daily_files:
    df = await _features_for_day(p)
    if df.empty:
        log.info('02 │ %s: empty (skipped)', Path(p).name); continue
    day_frames.append(df)
    log.info('02 │ %s: %d rows', Path(p).name, len(df))

all_df = pd.concat(day_frames, ignore_index=True)
log.info('02 │ multi-day feature frame: %s', all_df.shape)
assert len(day_frames) >= 3, 'Need ≥3 non-empty days'

## Section 03 — Labels + multi-day CPCV

In [ ]:
from p6lab.validation.labelers import triple_barrier_labels

# Triple-barrier labels per day (don't let labels straddle day boundaries).
# Each day produces +1/0/-1; drop timeouts before training.
HORIZON_MS = 60_000

y_list = []
for df in day_frames:
    mid = df['__mid__'].to_numpy()
    ts_arr = df['__ts_ms__'].to_numpy(dtype=np.int64)
    lbls = triple_barrier_labels(
        mid, ts_arr,
        horizon_ms=HORIZON_MS,
        up_target_ticks=4.0, down_target_ticks=4.0,
        tick_size=TICK_SIZE,
    )
    y_list.append(np.array([lbl.side for lbl in lbls], dtype=int))
y_all = np.concatenate(y_list)
valid = y_all != 0   # drop timeouts

# __-prefixed columns are metadata / label-source, never features
drop_cols = ['__ts_ms__', '__source_file__', '__mid__']
X = all_df.loc[valid, [c for c in all_df.columns if c not in drop_cols]].fillna(0).reset_index(drop=True)
y_arr = (y_all[valid] > 0).astype(int)
timestamps = pd.to_datetime(all_df.loc[valid, '__ts_ms__'].values, unit='ms').to_series().reset_index(drop=True)

# Trim constant columns
X = X.loc[:, X.std() > 0]
log.info('03 │ triple-barrier: %d up / %d down / %d timeout → X shape %s (%.2f%% bull)',
         (y_all ==  1).sum(), (y_all == -1).sum(), (y_all == 0).sum(),
         X.shape, 100 * y_arr.mean())

cv = CascadeAwareCPCV(
    n_splits=min(5, len(day_frames)), n_test_groups=2,
    trading_day_aware=True, cascade_embargo_days=1,
    min_train_days=1, min_test_days=1,
)
folds = cv.split(X, timestamps, cascade_timestamps=None)
log.info('03 │ built %d CPCV folds (day-aware)', len(folds))
for f in folds:
    tr_days = timestamps.iloc[f.train_idx].dt.normalize().nunique()
    te_days = timestamps.iloc[f.test_idx].dt.normalize().nunique()
    log.info('      fold %d: train_days=%d test_days=%d  (train=%d test=%d rows)',
             f.fold_id, tr_days, te_days, len(f.train_idx), len(f.test_idx))
assert folds, 'no valid CPCV folds'

## Section 04 — LightGBM per-fold training + per-day residuals

In [5]:
aucs, per_fold_info = [], []
day_residuals = {}   # day -> list of (y_true, y_pred) across folds

for f in folds:
    tr_x, tr_y = X.iloc[f.train_idx], y_arr[f.train_idx]
    te_x, te_y = X.iloc[f.test_idx],  y_arr[f.test_idx]
    if len(np.unique(tr_y)) < 2 or len(np.unique(te_y)) < 2:
        log.info('04 │ fold %d: skipped (single-class)', f.fold_id); continue
    model = lgb.LGBMClassifier(n_estimators=100, max_depth=6, random_state=42, verbose=-1)
    model.fit(tr_x, tr_y)
    proba = model.predict_proba(te_x)[:, 1]
    auc = roc_auc_score(te_y, proba)
    aucs.append(auc)
    per_fold_info.append({'fold': f.fold_id, 'auc': float(auc),
                          'n_train': len(tr_x), 'n_test': len(te_x)})

    # Per-day residuals
    test_days = timestamps.iloc[f.test_idx].dt.normalize().values
    for d, y_t, y_p in zip(test_days, te_y, proba):
        day_residuals.setdefault(pd.Timestamp(d), []).append((int(y_t), float(y_p)))

lgbm_mean_auc = float(np.mean(aucs)) if aucs else float('nan')
log.info('04 │ multi-day CPCV AUC mean=%.3f across %d folds', lgbm_mean_auc, len(aucs))
fold_df = pd.DataFrame(per_fold_info)
fold_df

04 │ multi-day CPCV AUC mean=▒.▒▒ across 6 folds


,fold,auc,n_train,n_test
0,0,▒.▒▒,2701,2899
1,1,▒.▒▒,4012,1588
2,2,▒.▒▒,4369,1231
3,3,▒.▒▒,1231,4369
4,4,▒.▒▒,1588,4012
5,5,▒.▒▒,2899,2701


## Section 05 — Per-day AUC (residual analysis)

In [ ]:
rows = []
for d, pairs in sorted(day_residuals.items()):
    yt = np.array([p[0] for p in pairs])
    yp = np.array([p[1] for p in pairs])
    if len(np.unique(yt)) < 2:
        rows.append({'day': str(d.date()), 'n': len(yt), 'auc': float('nan'),
                     'mean_pred': float(yp.mean()),
                     'note': 'single_class_test_set'})
        continue
    rows.append({'day': str(d.date()), 'n': len(yt),
                 'auc': float(roc_auc_score(yt, yp)),
                 'mean_pred': float(yp.mean()),
                 'note': ''})
per_day_df = pd.DataFrame(rows)
log.info('05 │ per-day AUC:\n%s', per_day_df.to_string(index=False))
# Audit item D: at least 2 days must produce a valid AUC (reject silent NaN sprawl)
assert per_day_df['auc'].dropna().shape[0] >= 2, (
    f'§05 at least 2 days must produce a valid AUC '
    f'(got {per_day_df["auc"].dropna().shape[0]}/{len(per_day_df)})'
)
per_day_df

## Section 06 — Must-beat-baseline gate

In [7]:
feat_name = 'imbalance_ema' if 'imbalance_ema' in X.columns else X.columns[0]
lr = LogisticRegression(max_iter=500).fit(X[[feat_name]], y_arr)
baseline_auc = float(roc_auc_score(y_arr, lr.predict_proba(X[[feat_name]])[:, 1]))

report = must_beat_baseline(
    candidate_metric=lgbm_mean_auc,
    baseline_metric=baseline_auc,
    min_improvement=BASELINE_MIN_AUC_IMPROVEMENT,
)
BASELINE_GATE_PASSED = bool(report.approved)
ALLOW_EXPORT_OVERRIDE = False
log.info('06 │ LightGBM %.3f vs baseline %.3f → %s', lgbm_mean_auc, baseline_auc, report.reason)
print(f'BASELINE_GATE_PASSED={BASELINE_GATE_PASSED}  '
      f'(Δ={lgbm_mean_auc - baseline_auc:+.4f}, required≥{BASELINE_MIN_AUC_IMPROVEMENT:+.4f})')

06 │ LightGBM ▒.▒▒ vs baseline ▒.▒▒ → approved: improvement ▒.▒▒ ≥ ▒.▒▒ and CI lower ▒.▒▒ > 0


BASELINE_GATE_PASSED=True  (Δ=+▒.▒▒, required≥+▒.▒▒)


## Section 07 — Gated export + registry promotion

In [ ]:
if not BASELINE_GATE_PASSED and not ALLOW_EXPORT_OVERRIDE:
    raise RuntimeError(
        f'§07 │ EXPORT BLOCKED — multi-day LightGBM ({lgbm_mean_auc:.3f}) failed to beat '
        f'baseline ({baseline_auc:.3f}) by ≥{BASELINE_MIN_AUC_IMPROVEMENT:.0%}. '
        f'Set ALLOW_EXPORT_OVERRIDE=True in §06 to ship anyway (debug only).'
    )

from datetime import timezone
version = f'v2_nq_fwd1m_multiday_{len(day_frames)}d'
model_dir = _common.versioned_dir(
    MODELS_DIR, version, data_slice={'mode': 'replay', 'days': len(day_frames)},
    extra={'lgbm_cv_auc': lgbm_mean_auc, 'baseline_auc': baseline_auc,
           'n_folds': len(folds), 'n_days': len(day_frames),
           'n_snapshots_per_day': MAX_SNAPS_PER_DAY},
)

# Refit one final model on all data for export
final = lgb.LGBMClassifier(n_estimators=100, max_depth=6, random_state=42, verbose=-1)
final.fit(X, y_arr)
model_path = model_dir / f'{version}.pkl'
rng = np.random.default_rng(42)
median_row = X.median().to_numpy()
model_dict = {
    'version': version,
    'matcher_templates': {'multiday_bull': rng.normal(size=(10, BOOK_SHAPE_DIM))},
    'matcher_centroids': {'multiday_bull': median_row[:12] if len(median_row) >= 12 else median_row},
    'pattern_contexts':  {'multiday_bull': {'vix_regime': 'normal'}},
    'global_covariance': np.cov(X.to_numpy().T),
    # 'primary_model' — engine reload_model() also accepts legacy 'lightgbm_model'.
    'primary_model': pickle.dumps(final),
    'feature_names': list(X.columns),
    'cv_auc': lgbm_mean_auc,
    'baseline_auc': baseline_auc,
}
with open(model_path, 'wb') as f:
    pickle.dump(model_dict, f)
log.info('07 │ wrote %s (%.1f KB)', model_path, model_path.stat().st_size / 1024)

# Promote CURRENT.json
registry_path = MODELS_DIR / 'CURRENT.json'
rel_model_path = model_path.relative_to(registry_path.parent)
registry_entry = {
    'version': version,
    'model_path': str(rel_model_path),
    'promoted_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'promoted_by': 'nb-multiday-baseline-gate',
    'lgbm_cv_auc': lgbm_mean_auc,
    'baseline_auc': baseline_auc,
    'n_days': len(day_frames),
}
with open(registry_path, 'w') as f:
    json.dump(registry_entry, f, indent=2)
log.info('07 │ promoted to CURRENT.json → %s', rel_model_path)
log.info('07 │ artefacts dir: %s', model_dir)